In [ ]:
# Menyatukan katalog earthquake dari csv USGS #
import pandas as pd
import os
import glob

# 1. Tentukan path folder tempat file CSV berada
folder_path = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/usgs_katalog/earthquake_csv'
output_file = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/usgs_katalog/katalog_usgs_master_2001_2025.csv'

# 2. Ambil semua list file yang berakhiran .csv di dalam folder tersebut
# Menggunakan glob agar urutan file bisa kita kontrol
csv_files = glob.glob(os.path.join(folder_path, "earthquake_*.csv"))
csv_files.sort()  # Mengurutkan berdasarkan nama file (tahun)

print(f"Ditemukan {len(csv_files)} file katalog. Memulai penggabungan...")

# 3. List untuk menampung dataframe
df_list = []

for file in csv_files:
    print(f"Membaca: {os.path.basename(file)}")
    # Membaca data, pastikan kolom waktu terbaca sebagai datetime
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

# 4. Gabungkan semua dataframe menjadi satu
df_master = pd.concat(df_list, ignore_index=True)

# 5. Opsional: Pembersihan data (Sangat disarankan untuk pengunduhan waveform)
# Mengonversi kolom waktu ke format datetime agar mudah diproses ObsPy
if 'time' in df_master.columns:
    df_master['time'] = pd.to_datetime(df_master['time'])
    # Mengurutkan berdasarkan waktu dari yang terlama ke terbaru
    df_master = df_master.sort_values(by='time')

# 6. Simpan sebagai satu file CSV induk
df_master.to_csv(output_file, index=False)
print(df_master.isnull().sum())
print("-" * 30)
print(f"✅ Sukses! Katalog master disimpan di: {output_file}")
print(f"Total kejadian gempa: {len(df_master)} event")

In [ ]:
# melihat data gempa tektonik dengan visualisasi sederhana #
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))

# Plot 1: Distribusi Kedalaman (Gempa tektonik biasanya punya pola cluster)
plt.subplot(1, 2, 1)
sns.histplot(df_master['depth'], bins=50, kde=True, color='teal')
plt.title('Distribusi Kedalaman (Depth)')
plt.xlabel('Kedalaman (km)')

# Plot 2: Hubungan Magnitude vs Kedalaman
plt.subplot(1, 2, 2)
plt.scatter(df_master['mag'], df_master['depth'], alpha=0.1, s=1)
plt.gca().invert_yaxis() # Kedalaman nol di atas
plt.title('Magnitude vs Kedalaman')
plt.xlabel('Magnitude')
plt.ylabel('Depth (km)')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Identifikasi tipe yang ada di katalog
print("Daftar Tipe Kejadian di Katalog:")
print(df_master['type'].value_counts())

# 2. Filter hanya yang murni 'earthquake'
df_master_clean = df_master[df_master['type'] == 'earthquake'].copy()

# 3. Simpan katalog yang sudah bersih
output_clean = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/usgs_katalog/katalog_usgs_earthquake_only.csv'
df_master_clean.to_csv(output_clean, index=False)

print("-" * 30)
print(f"✅ Filter Selesai!")
print(f"Total awal: {len(df_master)}")
print(f"Total setelah dibersihkan (Earthquake Only): {len(df_master_clean)}")

In [ ]:
import pandas as pd
import os
import time
import logging
import warnings
from obspy.clients.fdsn import Client
from obspy import UTCDateTime, Stream
from tqdm import tqdm
from IPython.display import clear_output

# --- 1. KONFIGURASI PATH ---
BASE_SAVE_DIR = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/waveform_usgs_2001_2025'
CATALOG_PATH = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/usgs_katalog/katalog_usgs_master_2001_2025.csv'
SUCCESS_LOG = os.path.join(BASE_SAVE_DIR, 'download_success.csv')
FAILED_LOG = os.path.join(BASE_SAVE_DIR, 'download_failed.csv')
SYSTEM_LOG = os.path.join(BASE_SAVE_DIR, 'system_warnings.log')

# --- 2. SETUP LOGGING & WARNINGS ---
if not os.path.exists(BASE_SAVE_DIR):
    os.makedirs(BASE_SAVE_DIR)

# Alihkan UserWarnings (kuning) ke file system_warnings.log
logging.basicConfig(filename=SYSTEM_LOG, level=logging.WARNING, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

def warn_with_logging(message, category, filename, lineno, file=None, line=None):
    logging.warning(f"{category.__name__}: {message} ({filename}:{lineno})")

warnings.showwarning = warn_with_logging

# --- 3. CORE DOWNLOADER FUNCTION ---
def run_ultimate_downloader():
    # Inisialisasi Client
    client = Client("IRIS", timeout=180)
    BATCH_SIZE = 5

    # Baca Katalog
    df = pd.read_csv(CATALOG_PATH)
    df['time'] = pd.to_datetime(df['time'], format='ISO8601')
    df['year'] = df['time'].dt.year
    total_event = len(df)

    print(f"🚀 Misi Dimulai: Mengolah {total_event} event.")
    print(f"📊 Progress Bar akan diperbarui setiap batch.")
    print(f"📝 Log sistem dialihkan ke: {os.path.basename(SYSTEM_LOG)}")
    
    # Progress Bar Utama
    pbar = tqdm(total=total_event, 
                desc="📊 TOTAL PROGRESS", 
                unit="ev",
                ncols=110,
                bar_format='{l_bar}{bar:30}{r_bar}')

    batch_counter = 0
    
    for year, df_year in df.groupby('year'):
        year_dir = os.path.join(BASE_SAVE_DIR, str(year))
        os.makedirs(year_dir, exist_ok=True)

        for i in range(0, len(df_year), BATCH_SIZE):
            batch_counter += 1
            
            # Bersihkan layar setiap 20 batch agar tetap rapi
            if batch_counter % 20 == 0:
                clear_output(wait=True)
                print(f"📅 Tahun Berjalan: {year}")
                print(f"📁 Folder Output: {year_dir}")

            subset = df_year.iloc[i : i + BATCH_SIZE].copy()
            batch_id = f"{year}_{i}"
            file_name = f"batch_usgs_{batch_id}.mseed"
            output_path = os.path.join(year_dir, file_name)

            # Auto-Resume: Skip jika file sudah ada
            if os.path.exists(output_path):
                pbar.update(len(subset))
                continue

            # Buat list permintaan bulk
            bulk_list = []
            for t in subset['time']:
                try:
                    ot = UTCDateTime(t)
                    bulk_list.append(("*", "*", "*", "BH?", ot - 10, ot + 20))
                except: continue

            success = False
            error_reason = ""

            # Attempt Download
            try:
                st = client.get_waveforms_bulk(bulk_list)
                if len(st) > 0:
                    st.write(output_path, format="MSEED")
                    success = True
                else:
                    error_reason = "Empty Stream"
            except Exception as e:
                err_msg = str(e).lower()
                # Adaptive Recovery Mode
                if any(x in err_msg for x in ["too much", "timeout", "500", "413"]):
                    st_rec = Stream()
                    for req in bulk_list:
                        try:
                            tmp = client.get_waveforms_bulk([req])
                            st_rec += tmp
                            time.sleep(0.5)
                        except: continue
                    if len(st_rec) > 0:
                        st_rec.write(output_path, format="MSEED")
                        success = True
                else:
                    error_reason = str(e)

            # Update Label Progress Bar
            pbar.set_postfix({
                "Year": year, 
                "Batch": i, 
                "Status": "✅ OK" if success else "❌ FAIL"
            })
            
            # Logging CSV
            ts_now = time.strftime('%Y-%m-%d %H:%M:%S')
            if success:
                pd.DataFrame({'timestamp': [ts_now], 'batch': [batch_id], 'path': [output_path]}).to_csv(
                    SUCCESS_LOG, mode='a', header=not os.path.exists(SUCCESS_LOG), index=False
                )
            else:
                subset['fail_time'] = ts_now
                subset['error_reason'] = error_reason
                subset.to_csv(FAILED_LOG, mode='a', header=not os.path.exists(FAILED_LOG), index=False)

            pbar.update(len(subset))
            time.sleep(1.2) # Jeda agar tidak terkena ban/block

    pbar.close()
    print("\n✅ SELESAI TOTAL! Seluruh tahun telah diproses.")

# Jalankan Program
if __name__ == "__main__":
    try:
        run_ultimate_downloader()
    except KeyboardInterrupt:
        print("\n🛑 Dihentikan manual. Progres terakhir tersimpan, silakan jalankan lagi nanti untuk RESUME.")
    except Exception as e:
        print(f"\n❌ Terjadi kesalahan fatal: {e}")

In [ ]:
import pandas as pd
import os
import time
import logging
import warnings
from obspy.clients.fdsn import Client
from obspy import UTCDateTime, Stream
from tqdm import tqdm
from IPython.display import clear_output

# --- 1. KONFIGURASI PATH & PARAMETER ---
BASE_SAVE_DIR = r'D:\output_waveform'
CATALOG_PATH = r'D:\seismiccode\usgs_katalog\katalog_usgs_master_2001_2025.csv'
SYSTEM_LOG = os.path.join(BASE_SAVE_DIR, 'system_log_modern.log')
FAILED_LOG = os.path.join(BASE_SAVE_DIR, 'failed_log_modern.csv')
# List Jaringan Prioritas (Asia-Australia-Global)
PRIORITY_NETS = ["GE", "IA"]

# --- 2. SETUP LINGKUNGAN ---
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

logging.basicConfig(
    filename=SYSTEM_LOG,
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)

def warn_with_logging(message, category, filename, lineno, file=None, line=None):
    logging.warning(f"{category.__name__}: {message} ({filename}:{lineno})")

warnings.showwarning = warn_with_logging

# --- 3. FUNGSI DOWNLOADER UTAMA ---
def run_ultimate_modern_downloader():
    client = Client("EARTHSCOPE", timeout=180)

    # 1. Load Katalog & Filter (Tahun >= 2015 & Mag >= 4.5)
    print("⏳ Membaca dan memfilter katalog...")
    df_full = pd.read_csv(CATALOG_PATH)
    df_full['time'] = pd.to_datetime(df_full['time'], format='ISO8601')

    df = df_full[(df_full['mag'] >= 4.5) & (df_full['time'].dt.year >= 2015)].copy()
    df['year'] = df['time'].dt.year
    total_event = len(df)

    clear_output()
    print(f"🚀 MCU-QUAKE PHASE 1: MODERN ERA (TARGETED)")
    print(f"🎯 Target: {total_event} Event | Mag >= 4.5 | Tahun 2015-2025")
    print(f"📡 Jaringan: {', '.join(PRIORITY_NETS)}")
    print("-" * 70)

    pbar = tqdm(total=total_event, desc="📊 TOTAL PROGRESS", unit="ev", ncols=110)

    batch_counter = 0
    failed_rows = []

    for year, df_year in df.groupby('year'):
        year_dir = os.path.join(BASE_SAVE_DIR, str(year))
        os.makedirs(year_dir, exist_ok=True)

        for idx, row in df_year.iterrows():
            batch_counter += 1
            event_id = f"{row['time'].strftime('%Y%m%d_%H%M%S')}_M{row['mag']:.1f}"
            output_path = os.path.join(year_dir, f"ev_{event_id}.mseed")

            # Auto-Resume
            if os.path.exists(output_path):
                pbar.update(1)
                continue

            if batch_counter % 20 == 0:
                clear_output(wait=True)
                print(f"⚡ Sedang Memproses: Tahun {year} | Event: {event_id}")

            ot = UTCDateTime(row['time'])
            bulk = []
            for net in PRIORITY_NETS:
                bulk.append((net, "*", "*", "BH?", ot - 10, ot + 20))

            success = False
            try:
                st = client.get_waveforms_bulk(bulk)
                if len(st) > 0:
                    st.merge(method=1, fill_value='interpolate')
                    st.write(output_path, format="MSEED")
                    success = True
            except Exception as e:
                logging.error(f"Gagal download event {event_id}: {str(e)}")
                failed_rows.append({
                    "event_id": event_id,
                    "time": row['time'],
                    "mag": row['mag'],
                    "year": year,
                    "error": str(e)
                })

            pbar.update(1)
            status_icon = "✅" if success else "⚠️"
            pbar.set_postfix({"Year": year, "Last_Status": status_icon})
            time.sleep(0.8)

    pbar.close()

    if failed_rows:
        pd.DataFrame(failed_rows).to_csv(FAILED_LOG, index=False)

    print("\n✅ SELURUH PROSES SELESAI!")
    print(f"📁 Hasil tersimpan di: {BASE_SAVE_DIR}")
    if failed_rows:
        print(f"⚠️ {len(failed_rows)} event gagal, tercatat di: {FAILED_LOG}")

# --- 4. JALANKAN ---
if __name__ == "__main__":
    try:
        run_ultimate_modern_downloader()
    except KeyboardInterrupt:
        print("\n🛑 Dihentikan manual. Jalankan lagi nanti untuk melanjutkan (Resume).")


⚡ Sedang Memproses: Tahun 2025 | Event: 20251227_030016_M4.9


📊 TOTAL PROGRESS: 100%|██████████████████| 17432/17432 [12:23:13<00:00,  2.56s/ev, Year=2025, Last_Status=✅]


✅ SELURUH PROSES SELESAI!
📁 Hasil tersimpan di: D:\output_waveform
⚠️ 2 event gagal, tercatat di: D:\output_waveform\failed_log_modern.csv
